#### Importing Libraries

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid")

print("Model Explainability Libraries Imported Successfully")

Model Explainability Libraries Imported Successfully


#### Configuring Project Directories

In [2]:
project_dir = Path("..").resolve()

data_dir = project_dir / "data" / "processed"
models_dir = project_dir / "models"
artifacts_dir = project_dir / "artifacts"
reports_dir = project_dir / "reports"
figures_dir = reports_dir / "figures"

required_directories = {
    "Processed Data": data_dir,
    "Models": models_dir,
    "Artifacts": artifacts_dir,
    "Reports": reports_dir,
    "Figures": figures_dir
}

for name, directory in required_directories.items():
    assert directory.is_dir(), f"Directory not found: {directory}"
    print(f"{name}: {directory}")

print("\nProject Directories Configured Successfully")

Processed Data: D:\Projects\ML and Data Science\Retail Sales Dynamics\data\processed
Models: D:\Projects\ML and Data Science\Retail Sales Dynamics\models
Artifacts: D:\Projects\ML and Data Science\Retail Sales Dynamics\artifacts
Reports: D:\Projects\ML and Data Science\Retail Sales Dynamics\reports
Figures: D:\Projects\ML and Data Science\Retail Sales Dynamics\reports\figures

Project Directories Configured Successfully


#### Loading Trained Models and Configuration Artifacts

In [3]:
with open(
    artifacts_dir / "forecast_feature_config.json",
    "r",
    encoding="utf-8"
) as file:
    feature_config = json.load(file)

with open(
    artifacts_dir / "final_forecasting_configuration.json",
    "r",
    encoding="utf-8"
) as file:
    final_model_config = json.load(file)

with open(
    artifacts_dir / "model_evaluation_summary.json",
    "r",
    encoding="utf-8"
) as file:
    evaluation_summary = json.load(file)

extra_trees_models = joblib.load(
    models_dir / "final_extra_trees.joblib"
)

autoreg_model = joblib.load(
    models_dir / "final_autoreg.pkl"
)

sarima_model = joblib.load(
    models_dir / "final_sarima.pkl"
)

forecast_horizon = int(feature_config["forecast_horizon"])

assert forecast_horizon == 7
assert len(extra_trees_models) == forecast_horizon

print("Final Model:", final_model_config["model_name"])
print("Forecast Horizon:", forecast_horizon)
print("Extra Trees Models Loaded:", len(extra_trees_models))
print("AutoReg Model Loaded:", type(autoreg_model).__name__)
print("SARIMA Model Loaded:", type(sarima_model).__name__)
print("Evaluation Summary Loaded:", bool(evaluation_summary))

print("\nTrained Models and Configuration Artifacts Loaded Successfully")

Final Model: Voting_Hybrid
Forecast Horizon: 7
Extra Trees Models Loaded: 7
AutoReg Model Loaded: AutoRegResultsWrapper
SARIMA Model Loaded: SARIMAXResultsWrapper
Evaluation Summary Loaded: True

Trained Models and Configuration Artifacts Loaded Successfully


#### Loading Feature Datasets and Evaluation Results

In [4]:
train_df = pd.read_parquet(
    data_dir / "forecast_features_train.parquet"
)

test_df = pd.read_parquet(
    data_dir / "forecast_features_test.parquet"
)

daily_revenue_df = pd.read_parquet(
    data_dir / "daily_revenue.parquet"
)

rolling_predictions_df = pd.read_parquet(
    artifacts_dir / "rolling_test_predictions.parquet"
)

forecast_error_df = pd.read_parquet(
    artifacts_dir / "forecast_error_analysis.parquet"
)

horizon_performance_df = pd.read_csv(
    artifacts_dir / "horizon_wise_performance.csv"
)

test_performance_df = pd.read_csv(
    artifacts_dir / "test_model_performance.csv"
)

loaded_datasets = {
    "Training Features": train_df,
    "Test Features": test_df,
    "Daily Revenue": daily_revenue_df,
    "Rolling Predictions": rolling_predictions_df,
    "Forecast Errors": forecast_error_df,
    "Horizon Performance": horizon_performance_df,
    "Test Performance": test_performance_df
}

print("Loaded Dataset Dimensions:\n")

for name, dataframe in loaded_datasets.items():
    print(f"{name}: {dataframe.shape}")

assert len(train_df) == 207
assert len(test_df) == 55
assert len(daily_revenue_df) == 305
assert len(rolling_predictions_df) == 385

print("\nFeature Datasets and Evaluation Results Loaded Successfully")

Loaded Dataset Dimensions:

Training Features: (207, 87)
Test Features: (55, 87)
Daily Revenue: (305, 2)
Rolling Predictions: (385, 12)
Forecast Errors: (385, 16)
Horizon Performance: (42, 6)
Test Performance: (6, 7)

Feature Datasets and Evaluation Results Loaded Successfully


#### Verifying Horizon-Specific Feature Mappings

In [6]:
shared_features = feature_config["shared_features"]
horizon_features = feature_config["horizon_features"]
horizon_targets = feature_config["horizon_targets"]

print("Feature Configuration Structure:")
print("Shared Features Type:", type(shared_features).__name__)
print("Horizon Features Type:", type(horizon_features).__name__)
print("Horizon Feature Keys:", list(horizon_features.keys()))

print("\nHorizon-Specific Feature Diagnostics:\n")

diagnostic_records = []

for horizon in range(1, forecast_horizon + 1):
    model = extra_trees_models[horizon]
    horizon_key = str(horizon)

    configured_features = (
        shared_features + horizon_features[horizon_key]
    )

    model_features = list(
        getattr(model, "feature_names_in_", [])
    )

    diagnostic_records.append({
        "Horizon": horizon,
        "Shared_Features": len(shared_features),
        "Horizon_Features": len(horizon_features[horizon_key]),
        "Configured_Total": len(configured_features),
        "Model_Expected": model.n_features_in_,
        "Model_Feature_Names": len(model_features),
        "Count_Match": (
            len(configured_features) == model.n_features_in_
        )
    })

feature_diagnostic_df = pd.DataFrame(diagnostic_records)

display(feature_diagnostic_df)

print("\nHorizon 1 — Configured Feature Names:")
print(configured_features if forecast_horizon == 1 else (
    shared_features + horizon_features["1"]
))

print("\nHorizon 1 — Model Feature Names:")
print(
    list(
        getattr(
            extra_trees_models[1],
            "feature_names_in_",
            []
        )
    )
)

print("\nFeature Mapping Diagnostics Completed")

Feature Configuration Structure:
Shared Features Type: list
Horizon Features Type: dict
Horizon Feature Keys: ['1', '2', '3', '4', '5', '6', '7']

Horizon-Specific Feature Diagnostics:



,Horizon,Shared_Features,Horizon_Features,Configured_Total,Model_Expected,Model_Feature_Names,Count_Match
0,1,44,48,92,48,48,False
1,2,44,48,92,48,48,False
2,3,44,48,92,48,48,False
3,4,44,48,92,48,48,False
4,5,44,48,92,48,48,False
5,6,44,48,92,48,48,False
6,7,44,48,92,48,48,False



Horizon 1 — Configured Feature Names:
['Origin_Revenue', 'Revenue_Lag_1', 'Revenue_Lag_2', 'Revenue_Lag_3', 'Revenue_Lag_6', 'Revenue_Lag_7', 'Revenue_Lag_12', 'Revenue_Lag_14', 'Revenue_Lag_18', 'Revenue_Lag_24', 'Revenue_Lag_30', 'Revenue_Rolling_Mean_3', 'Revenue_Rolling_Std_3', 'Revenue_Rolling_Min_3', 'Revenue_Rolling_Max_3', 'Revenue_Rolling_Mean_6', 'Revenue_Rolling_Std_6', 'Revenue_Rolling_Min_6', 'Revenue_Rolling_Max_6', 'Revenue_Rolling_Mean_7', 'Revenue_Rolling_Std_7', 'Revenue_Rolling_Min_7', 'Revenue_Rolling_Max_7', 'Revenue_Rolling_Mean_14', 'Revenue_Rolling_Std_14', 'Revenue_Rolling_Min_14', 'Revenue_Rolling_Max_14', 'Revenue_Rolling_Mean_30', 'Revenue_Rolling_Std_30', 'Revenue_Rolling_Min_30', 'Revenue_Rolling_Max_30', 'Revenue_Change_1', 'Revenue_Change_6', 'Revenue_Change_12', 'Revenue_Momentum_3_7', 'Revenue_Momentum_7_30', 'Revenue_Ratio_3_7', 'Revenue_Ratio_7_30', 'Origin_Day_Of_Week', 'Origin_Day_Of_Month', 'Origin_Month', 'Origin_Quarter', 'Origin_Day_Of_Year', 

#### Verifying Horizon-Specific Feature Mappings

In [7]:
horizon_features = feature_config["horizon_features"]
horizon_targets = feature_config["horizon_targets"]

horizon_feature_columns = {}
horizon_target_columns = {}
feature_verification_records = []

for horizon in range(1, forecast_horizon + 1):
    horizon_key = str(horizon)

    feature_columns = horizon_features[horizon_key]
    target_column = horizon_targets[horizon_key]
    model = extra_trees_models[horizon]

    assert len(feature_columns) == len(set(feature_columns)), (
        f"Horizon {horizon}: Duplicate features detected"
    )

    assert all(
        column in train_df.columns and column in test_df.columns
        for column in feature_columns
    ), f"Horizon {horizon}: Missing feature columns"

    assert target_column in train_df.columns
    assert target_column in test_df.columns

    assert len(feature_columns) == model.n_features_in_, (
        f"Horizon {horizon}: Feature count mismatch"
    )

    assert list(model.feature_names_in_) == feature_columns, (
        f"Horizon {horizon}: Feature order mismatch"
    )

    horizon_feature_columns[horizon] = feature_columns
    horizon_target_columns[horizon] = target_column

    feature_verification_records.append({
        "Horizon": horizon,
        "Feature_Count": len(feature_columns),
        "Training_Origins": len(train_df),
        "Test_Origins": len(test_df),
        "Target": target_column,
        "Status": "Verified"
    })

feature_verification_df = pd.DataFrame(
    feature_verification_records
)

print("Horizon-Specific Feature Verification:\n")

display(feature_verification_df)

print("\nAll Horizon-Specific Feature Mappings Verified Successfully")

Horizon-Specific Feature Verification:



,Horizon,Feature_Count,Training_Origins,Test_Origins,Target,Status
0,1,48,207,55,Target_H1,Verified
1,2,48,207,55,Target_H2,Verified
2,3,48,207,55,Target_H3,Verified
3,4,48,207,55,Target_H4,Verified
4,5,48,207,55,Target_H5,Verified
5,6,48,207,55,Target_H6,Verified
6,7,48,207,55,Target_H7,Verified



All Horizon-Specific Feature Mappings Verified Successfully


**Key Observations**
- **Feature Consistency:** All seven Extra Trees models use 48 verified input features, with feature names and ordering matching their respective training configurations.
- **Horizon-Specific Modeling:** Each forecasting horizon has a dedicated model trained on 207 forecast origins, with 55 test origins available for evaluating feature contributions.

#### Calculating Extra Trees Feature Importance

In [8]:
feature_importance_records = []

for horizon in range(1, forecast_horizon + 1):
    model = extra_trees_models[horizon]
    feature_columns = horizon_feature_columns[horizon]

    importances = model.feature_importances_

    assert len(importances) == len(feature_columns)
    assert np.isclose(importances.sum(), 1.0)

    for feature, importance in zip(
        feature_columns,
        importances
    ):
        feature_importance_records.append({
            "Horizon": horizon,
            "Feature": feature,
            "Importance": float(importance)
        })

feature_importance_df = pd.DataFrame(
    feature_importance_records
)

average_feature_importance_df = (
    feature_importance_df
    .groupby("Feature", as_index=False)
    .agg(
        Mean_Importance=("Importance", "mean"),
        Horizons_Used=("Horizon", "nunique")
    )
    .sort_values(
        "Mean_Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Feature Importance Records:", len(feature_importance_df))
print("Unique Features:", feature_importance_df["Feature"].nunique())

print("\nTop 15 Features by Average Importance:\n")

display(
    average_feature_importance_df.head(15).style.format({
        "Mean_Importance": "{:.4f}"
    })
)

print("\nExtra Trees Feature Importance Calculation Completed")

Feature Importance Records: 336
Unique Features: 72

Top 15 Features by Average Importance:



,Feature,Mean_Importance,Horizons_Used
0,Target_Is_Weekend_H2,0.3773,1
1,Target_Is_Weekend_H1,0.3761,1
2,Target_Is_Weekend_H3,0.3579,1
3,Target_Is_Weekend_H5,0.3504,1
4,Target_Is_Weekend_H6,0.3468,1
5,Target_Is_Weekend_H4,0.3434,1
6,Target_Is_Weekend_H7,0.3392,1
7,Target_Day_Of_Week_H1,0.0916,1
8,Target_Day_Of_Week_H2,0.0864,1
9,Target_Day_Of_Week_H3,0.0796,1



Extra Trees Feature Importance Calculation Completed


**Key Observations**
- **Weekend Influence:** Target-day weekend indicators dominate Extra Trees feature importance across all seven horizons, contributing approximately 33.92%–37.73% of each model's total impurity-based importance.
- **Calendar-Based Forecasting:** Target day-of-week features also rank prominently, with importance scores ranging from 0.0697 to 0.0916, indicating substantial reliance on trading-calendar information.